# 04 · Preliminary run of the learners — Neonatal mortality NCHS (M4)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 → 02: S4 · G1 → 03: S5 · S6 · S7 · G2 · G3 → **04: S8** → 05: S8 · S9 · G4 → 06: S10 · S11 |
| What it does | Compares logistic regression, random forest, extra trees, XGBoost and a stacked ensemble with the same hyperparameter search, and applies the primary-learner rule |
| Stages | S8 (preliminary run) |
| Design | Cohorts 2016–2021; all deaths and 2% of other births, weighted; 80/20 split; 20 combinations per learner; median imputation for all (M4-D3) |
| Inputs | Contract table (notebook 03) |
| Outputs | `data/outputs_04_preliminary_learners.txt`, `data/outputs_04_selection.json` (read by notebook 05), `figures/04_roc_learners.png` |
| Time and memory | 1 to 3 hours and ≈8 GB; each finished learner is saved in `data/checkpoints_04/` and loaded on a new run |
| Previous / next | `03_predictors_g2_g3` / `05_main_evaluation` |

> **Running again from scratch.** Each finished learner is saved in `data/checkpoints_04/` and loaded instead of being refitted. To repeat the whole search (for example, if the contract table was rebuilt), delete that folder before running.

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_04_preliminary_learners.txt"

# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")


SEED = 20261123
DEV_YEARS = list(range(2016, 2022))       # 2022-2023 are kept untouched for the temporal test (C4)
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))   # share of non-events sampled; all events kept
QUICK = os.environ.get("NCHS_QUICK") == "1"               # only to test the code on a small sample
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
# Predictors: every column of the contract table (notebook 03) except identifiers, labels and data-quality flags
import pyarrow.parquet as pq
META = ["unit_id", "group_id", "period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal",
        "ge28", "ga_imputed", "bw_imputed", "n_unknown"]
PREDICTORS = [c for c in pq.read_schema(CONTRACT_DIR / "cohort_2021.parquet").names if c not in META]
assert len(PREDICTORS) == 71, len(PREDICTORS)
SELECTION_FILE = ROOT / "data" / "outputs_04_selection.json"   # read by notebook 05 (hyperparameters only)
lines = []

def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)
SUPPRESS_SMALL = False
import contextlib, io, html as _html
_show_en = show

def _fmt(v):
    # Display format of a table cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if SUPPRESS_SMALL and isinstance(v, (int, np.integer)) and 0 < v < 10:
        return "<10"
    if isinstance(v, (int, np.integer)) or (isinstance(v, (float, np.floating)) and float(v).is_integer() and abs(v) >= 1000):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _render(title, obj):
    # In Jupyter: a heading and an HTML table with scroll; elsewhere: the text block.
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
    except ImportError:
        return print(lines_es[-1] if LANG == "es" else lines[-1])
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(str(title))}</b></p>"
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        if isinstance(obj, pd.Series):          # rows are different quantities: each cell on its own
            df = obj.astype(object).map(_fmt).to_frame(name=translate("value", LANG))
        else:                                    # columns: the same number of decimals down each column
            df = obj.copy().astype(object)
            for k in range(df.shape[1]):
                col = obj.iloc[:, k]
                if pd.api.types.is_float_dtype(col) and not col.dropna().map(float.is_integer).all():
                    dec = max((len(f"{v:.4g}".split(".")[1]) if "." in f"{v:.4g}" and "e" not in f"{v:.4g}" else 0)
                              for v in col.dropna()) if col.notna().any() else 0
                    df.iloc[:, k] = col.map(lambda v, d=dec: "" if pd.isna(v) else f"{v:,.{d}f}")
                else:
                    df.iloc[:, k] = col.map(_fmt)
        table = df.to_html(na_rep="", border=0)
        _display(HTML(head + f"<div style='max-height:420px; overflow:auto'>{table}</div>"))
    else:
        _display(HTML(head + f"<pre style='margin:0'>{_html.escape(str(obj))}</pre>"))

def show(title, obj):
    # The English block is kept in `lines` (data/outputs_*.txt) and, in Spanish, the translated block in
    # `lines_es` (data/outputs_*_es.txt); on screen the table is shown as an HTML table.
    with contextlib.redirect_stdout(io.StringIO()):
        _show_en(title, obj)
        if LANG == "es":
            _show_en(translate(title, "es"), translate(obj, "es"))
            lines_es.append(lines.pop())
    _render(translate(title, LANG), translate(obj, LANG))

## E8 · Modeling — development sample and 80/20 split

The weighted case-control sample stands for the whole population; the AUROC does not depend on that sampling.

In [ ]:
# Development sample: every event and a random FRACTION of non-events of 2016-2021, with weights
# 1/FRACTION for non-events so that weighted metrics and calibration refer to the full population.
rng = np.random.default_rng(SEED)
parts, n_dev, e_dev = [], 0, 0
for y in DEV_YEARS:
    # read in batches and keep the sample only, so the memory stays low
    for b in pq.ParquetFile(CONTRACT_DIR / f"cohort_{y}.parquet").iter_batches(500_000, columns=PREDICTORS + ["label_main"]):
        c = b.to_pandas()
        n_dev += len(c); e_dev += int(c["label_main"].sum())
        keep = c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)
        parts.append(c[keep])
s = pd.concat(parts, ignore_index=True); del parts
if QUICK:
    s = s.sample(n=min(len(s), 40000), random_state=SEED)
y = s.pop("label_main").to_numpy(bool)
w = np.where(y, 1.0, 1.0 / FRACTION)
X = s.astype("float32")

# Preliminary run (framework, S8): one stratified split of the development sample, 80% to fit and
# tune (inner 3-fold search), 20% to compare learners. Births are independent units (no mother identifier).
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=SEED)
show("Development data", pd.Series(dtype=object, data={
    "development births (2016-2021)": n_dev, "development events": e_dev,
    "prevalence": round(e_dev / n_dev, 5), f"sample: events + {100 * FRACTION:.0f}% of non-events": len(X),
    "fit/tune part": len(X_tr), "comparison part": len(X_te), "events in comparison part": int(y_te.sum())}))

## E8 · Modeling — hyperparameter search, ensemble and leakage diagnostic

Same budget for every learner. XGBoost with its own missing-value handling is fitted only as a diagnostic and cannot be selected.

| Result | What is done | In this model |
|---|---|---|
| Native XGBoost ≈ imputed XGBoost | Unknowns carry no relevant signal | +0.002 |
| Clearly above | Leakage signal from unknowns: reviewed at G2 before going on (as in the perinatal model) | — |

In [ ]:
# Hyperparameter search: random search with the SAME budget for every learner: N_CONFIGS
# combinations drawn from prespecified ranges, scored by AUROC in 3 inner folds of a stratified 30%
# subsample of the fit part; the best combination is then refitted on the whole fit part.
# Every learner imputes the median inside training (decision M4-D3). XGBoost: number of trees chosen by
# early stopping (up to 2,000, patience 50) in each inner fold; the refit uses their mean.
import gc, pickle, warnings
from sklearn.base import clone
warnings.filterwarnings("ignore", message=".*penalty.*")      # scikit-learn >= 1.8 deprecation notice for LR
CKPT_DIR = ROOT / "data" / "checkpoints_04"                     # local only (data/ is not versioned)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
N_CONFIGS = 3 if QUICK else 20
SEARCH_SHARE = 0.30
rs = np.random.default_rng(SEED)
loguni = lambda lo, hi: float(np.exp(rs.uniform(np.log(lo), np.log(hi))))

def sample_config(name):
    if name == "LR":
        return {"C": loguni(1e-4, 10), "penalty": str(rs.choice(["l1", "l2"]))}
    if name in ("RF", "ET"):
        return {"max_depth": [8, 12, 16, 20, 30, None][rs.integers(6)],
                "min_samples_leaf": int(loguni(20, 1000)),
                "max_features": ["sqrt", 0.3, 0.5][rs.integers(3)]}
    return {"max_depth": int(rs.integers(3, 11)), "learning_rate": loguni(0.01, 0.2),
            "min_child_weight": loguni(1, 100), "subsample": float(rs.uniform(0.6, 1.0)),
            "colsample_bytree": float(rs.uniform(0.6, 1.0)), "reg_lambda": loguni(0.1, 10),
            "reg_alpha": loguni(1e-3, 1)}

def build(name, cfg, n_estimators=None):
    imp = SimpleImputer(strategy="median")
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=n_estimators or (100 if QUICK else 2000), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, eval_metric="auc", **cfg))

def fit_w(est, X_, y_, w_, **extra):
    step = est.steps[-1][0]
    return est.fit(X_, y_, **{f"{step}__sample_weight": w_, **{f"{step}__{k}": v for k, v in extra.items()}})

X_se, _, y_se, _, w_se, _ = train_test_split(X_tr, y_tr, w_tr, train_size=SEARCH_SHARE, stratify=y_tr, random_state=SEED)
inner = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_se, y_se))
fitted, pred, grid_rows, trials = {}, {}, [], []
CONFIGS = {name: [sample_config(name) for _ in range(N_CONFIGS)] for name in ["LR", "RF", "ET", "XGB"]}
for name in ["LR", "RF", "ET", "XGB"]:
    ck = CKPT_DIR / f"{name}_seed{SEED}_frac{FRACTION}_quick{int(QUICK)}.pkl"
    if ck.exists():
        r = pickle.loads(ck.read_bytes())
        pred[name], fitted[name] = r["pred"], build(name, r["cfg"], n_estimators=r["n_iter"])
        grid_rows.append(r["grid_row"]); trials += r["trials"]
        print(name, "loaded from checkpoint", ck.name)
        continue
    t = time.time(); best = (-1, None, None)
    for k in range(N_CONFIGS):
        cfg = CONFIGS[name][k]; aucs, iters = [], []
        for tr, va in inner:
            est = build(name, cfg)
            if name == "XGB":
                imp = SimpleImputer(strategy="median").fit(X_se.iloc[tr])
                xgb = est.steps[-1][1].set_params(early_stopping_rounds=50)
                xgb.fit(imp.transform(X_se.iloc[tr]), y_se[tr], sample_weight=w_se[tr],
                        eval_set=[(imp.transform(X_se.iloc[va]), y_se[va])], verbose=False)
                p = xgb.predict_proba(imp.transform(X_se.iloc[va]))[:, 1]; iters.append(xgb.best_iteration + 1)
            else:
                p = fit_w(est, X_se.iloc[tr], y_se[tr], w_se[tr]).predict_proba(X_se.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_se[va], p))
        score = float(np.mean(aucs))
        trials.append({"learner": name, "inner AUROC": score, **{k: (round(v, 4) if isinstance(v, float) else str(v)) for k, v in cfg.items()}})
        if score > best[0]:
            best = (score, cfg, int(np.mean(iters)) if iters else None)
    score, cfg, n_iter = best
    m = fit_w(build(name, cfg, n_estimators=n_iter), X_tr, y_tr, w_tr)
    pred[name] = m.predict_proba(X_te)[:, 1]
    del m; gc.collect()                                   # keep only the predictions, not the fitted model
    fitted[name] = build(name, cfg, n_estimators=n_iter)  # unfitted copy, used by the ensemble and the diagnostic
    shown = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in cfg.items()}
    if n_iter: shown["n_estimators"] = n_iter
    grid_rows.append({"learner": name, "best combination": json.dumps(shown), "inner AUROC": round(score, 4),
                      "minutes": round((time.time() - t) / 60, 1)})
    ck.write_bytes(pickle.dumps({"cfg": cfg, "n_iter": n_iter, "pred": pred[name], "grid_row": grid_rows[-1],
                                 "trials": [x for x in trials if x["learner"] == name]}))
    print(name, "done in", grid_rows[-1]["minutes"], "min")
show(f"Random search ({N_CONFIGS} combinations per learner, 3 inner folds on {int(SEARCH_SHARE * 100)}% of the fit part)",
     pd.DataFrame(grid_rows).set_index("learner"))
show("Spread of the inner AUROC over the combinations tried (how much the search mattered)",
     pd.DataFrame(trials).groupby("learner")["inner AUROC"].agg(["min", "median", "max"]).round(4))
trials_df = pd.DataFrame(trials)
for name in ["LR", "RF", "ET", "XGB"]:
    t_ = trials_df[trials_df["learner"] == name].dropna(axis=1, how="all").drop(columns="learner")
    show(f"All combinations tried: {name}", t_.sort_values("inner AUROC", ascending=False).round(4).reset_index(drop=True))

# Stacked ensemble with a logistic meta-learner (comparison only): out-of-fold predictions of each
# tuned base learner inside the fit part train the meta-learner; the base learners fitted on the
# whole fit part score the comparison part.
t = time.time()
oof = np.zeros((len(X_tr), len(fitted)))
for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_tr, y_tr):
    for j, est in enumerate(fitted.values()):
        m = fit_w(clone(est), X_tr.iloc[tr], y_tr[tr], w_tr[tr])
        oof[va, j] = m.predict_proba(X_tr.iloc[va])[:, 1]
        del m; gc.collect()
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
meta = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))   # meta-learner on the logit scale
meta.fit(logit(oof), y_tr, logisticregression__sample_weight=w_tr)
pred["Stack"] = meta.predict_proba(logit(np.column_stack([pred[k] for k in fitted])))[:, 1]
print("Stack done in", round((time.time() - t) / 60, 1), "min")

# Leakage diagnostic (not eligible as primary): XGBoost with its NATIVE handling of missing values,
# which can route unknown values to their own branch and so learn from how completely the certificate
# was filled in (number of unknowns alone: AUROC 0.601, notebook 03).
native = clone(fitted["XGB"].steps[-1][1]).fit(X_tr, y_tr, sample_weight=w_tr)   # same tuned hyperparameters
pred["XGB native missing (diagnostic)"] = native.predict_proba(X_te)[:, 1]

## E8 · Primary-learner rule — comparison of the learners on the held-out 20% and selection

The rule was written before seeing results.

| Result | What is done | In this model |
|---|---|---|
| Learner with the highest AUROC | It is the primary, unless a simpler one is within 0.005 (order: LR, XGBoost, RF and ET) | XGBoost 0.964; LR 0.958 (−0.006) |
| Ensemble ≥ 0.005 above the primary with 95% CI > 0 | The ensemble is adopted | No: −0.0007 |

In [ ]:
# Comparison on the 20% part. AUROC does not depend on sampling the non-events; AUPRC and the
# observed/expected ratio use the weights so they refer to the full population.
B = 50 if QUICK else 300
boot = np.random.default_rng(SEED)
idx_pos, idx_neg = np.where(y_te)[0], np.where(~y_te)[0]
boot_idx = [np.concatenate([boot.choice(idx_pos, len(idx_pos)), boot.choice(idx_neg, len(idx_neg))]) for _ in range(B)]

def auroc(p, ix=None):
    ix = slice(None) if ix is None else ix
    return roc_auc_score(y_te[ix], p[ix])

rows = {}
for name, p in pred.items():
    a = [auroc(p, ix) for ix in boot_idx]
    o = np.argsort(p); cw = np.cumsum(w_te[o])
    cut = p[o][np.searchsorted(cw, 0.90 * cw[-1])]          # weighted 90th percentile of risk
    top = p >= cut
    rows[name] = {"AUROC": round(auroc(p), 4), "AUROC 2.5%": round(np.percentile(a, 2.5), 4),
                  "AUROC 97.5%": round(np.percentile(a, 97.5), 4),
                  "AUPRC (weighted)": round(average_precision_score(y_te, p, sample_weight=w_te), 4),
                  "observed/expected": round(float(y_te.sum() / np.sum(w_te * p)), 3),
                  "events in top 10% (%)": round(100 * float(y_te[top].sum() / y_te.sum()), 1)}
res = pd.DataFrame(rows).T
show(f"Preliminary comparison on the 20% part (prevalence {y_te.sum() / w_te.sum():.5f})", res)

# Prespecified selection rule (outcome card, S8): primary = best base learner by AUROC; when the
# difference with a simpler learner is under 0.005, the simpler one is preferred (LR < XGB < RF/ET).
order = ["LR", "XGB", "RF", "ET"]
best = max(order, key=lambda k: res.loc[k, "AUROC"])
primary = next(k for k in order if res.loc[best, "AUROC"] - res.loc[k, "AUROC"] < 0.005)
d_stack = [auroc(pred["Stack"], ix) - auroc(pred[primary], ix) for ix in boot_idx]
lo, hi = np.percentile(d_stack, [2.5, 97.5])
adopt = (res.loc["Stack", "AUROC"] - res.loc[primary, "AUROC"] >= 0.005) and lo > 0
show("Selection", pd.Series({"best base learner by AUROC": best, "primary learner (rule)": primary,
                             "Stack - primary AUROC": f"{np.mean(d_stack):+.4f} ({lo:+.4f} to {hi:+.4f})",
                             "ensemble adopted": adopt}))
# Accuracy and confusion matrices (requested in addition to the framework metrics). A threshold is
# needed: the birth is flagged when its risk is in the top 10% (and, second, the top 5%) of the
# weighted risk distribution, the same cut the framework uses for the share of events. Counts are
# weighted, so they estimate the full population of the comparison part (about 20% of 2016-2021).
def wcut(p, share):
    o = np.argsort(p); cw = np.cumsum(w_te[o])
    return p[o][np.searchsorted(cw, (1 - share) * cw[-1])]

cm_rows, prev_te = [], y_te.sum() / w_te.sum()
for share in (0.10, 0.05):
    for name, p in pred.items():
        flag = p >= wcut(p, share)
        tp = float(w_te[flag & y_te].sum()); fn = float(w_te[~flag & y_te].sum())
        fp = float(w_te[flag & ~y_te].sum()); tn = float(w_te[~flag & ~y_te].sum())
        n = tp + fn + fp + tn
        cm_rows.append({"flagged": f"top {int(share * 100)}%", "learner": name,
                        "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
                        "accuracy": round((tp + tn) / n, 4), "sensitivity": round(tp / (tp + fn), 3),
                        "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4),
                        "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrices and accuracy (weighted to the full population of the comparison part)",
     pd.DataFrame(cm_rows).set_index(["flagged", "learner"]))
show("Reference: accuracy of flagging nobody (all predicted without event)", round(1 - prev_te, 4))
write_outputs(OUT_FILE, "Preliminary run of the base learners (neonatal model, development 2016-2021)")
# Hyperparameters of the selected learners for notebook 05 (no record-level data)
BEST = {g["learner"]: json.loads(g["best combination"]) for g in grid_rows}
SELECTION_FILE.write_text(json.dumps({"primary": primary, "ensemble_adopted": bool(adopt), "best": BEST,
                                      "quick": QUICK}, indent=1), encoding="utf-8")
print("primary learner:", primary, "| written:", SELECTION_FILE)

## Figure · ROC curves of the learners

ROC curves on the comparison part. Accuracy, F1 and the confusion matrix go in the summary (cut: top 10% of risk); they do not decide in the framework.

In [ ]:
# Figure helpers (shared by the figure and summary cells). Curves use the out-of-fold or test predictions;
# confusion matrices are weighted to the full population (non-events carry weight 1/FRACTION).
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

def plot_roc(curves, title, fname):
    """curves: list of (label, y, p). One subplot per curve (algorithm or population), so the curves do not overlap.
    AUROC does not depend on the case-control weights."""
    n = len(curves); ncol = min(3, n); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.4 * nrow), squeeze=False)
    for k, (label, yy, pp) in enumerate(curves):
        ax = axes[k // ncol][k % ncol]
        fpr, tpr, _ = roc_curve(yy, pp)
        ax.plot(fpr, tpr, lw=1.8, color="#2F6DB5")
        ax.fill_between(fpr, tpr, alpha=0.08, color="#2F6DB5")
        ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
        ax.set_title(f"{translate(label, LANG)}\nAUROC {roc_auc_score(yy, pp):.3f}", fontsize=10)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
        ax.set_xlabel(translate("1 - specificity (false positive rate)", LANG), fontsize=9)
        ax.set_ylabel(translate("Sensitivity (true positive rate)", LANG), fontsize=9)
    for k in range(n, nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    fig.suptitle(translate(title, LANG), fontsize=12)
    fig.tight_layout(); fig.savefig(FIG_DIR / (fname.replace(".png", "_es.png") if LANG == "es" else fname), dpi=200); plt.show()

def cut_top(p, w, share):
    """Risk cut that flags the given share of the weighted population."""
    o = np.argsort(-p); cw = np.cumsum(w[o])
    return float(p[o][np.searchsorted(cw, share * cw[-1])])

def threshold_metrics(y, p, w, cut):
    """Accuracy, F1 and the confusion-matrix rates at a declared risk cut (weighted to the population)."""
    flag = p >= cut
    tp, fp = w[flag & y].sum(), w[flag & ~y].sum(); fn, tn = w[~flag & y].sum(), w[~flag & ~y].sum()
    sens, ppv = tp / (tp + fn), tp / (tp + fp)
    return {"risk cut": round(cut, 4), "TP": int(round(tp)), "FP": int(round(fp)), "FN": int(round(fn)), "TN": int(round(tn)),
            "accuracy": round(float((tp + tn) / (tp + fp + fn + tn)), 4), "F1": round(float(2 * sens * ppv / (sens + ppv)), 4),
            "sensitivity": round(float(sens), 4), "specificity": round(float(tn / (tn + fp)), 4),
            "PPV": round(float(ppv), 4), "NPV": round(float(tn / (tn + fn)), 5)}

def plot_cm(panels, fname):
    """panels: list of (title, y, p, w, cut). For each panel, on the left the classification report of the two classes
    (precision, recall, F1, accuracy and macro average) as a heatmap and, on the right, the confusion matrix. Counts are
    weighted to the full population (non-events carry weight 1/FRACTION); class 1 = neonatal death."""
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, axes = plt.subplots(len(panels), 2, figsize=(12.5, 4.6 * len(panels)), squeeze=False,
                             gridspec_kw={"width_ratios": [1.35, 1]})
    for row, (title, yy, pp, ww, cut) in enumerate(panels):
        m = threshold_metrics(yy, pp, ww, cut)
        tp, fp, fn, tn = m["TP"], m["FP"], m["FN"], m["TN"]
        f1 = lambda a, b: 2 * a * b / (a + b) if (a + b) else 0.0
        p0, r0 = tn / (tn + fn), tn / (tn + fp)
        p1, r1 = tp / (tp + fp), tp / (tp + fn)
        acc = (tp + tn) / (tp + fp + fn + tn)
        rep = np.array([[p0, r0, f1(p0, r0)], [p1, r1, f1(p1, r1)], [acc, acc, acc],
                        [(p0 + p1) / 2, (r0 + r1) / 2, (f1(p0, r0) + f1(p1, r1)) / 2]])
        ax = axes[row][0]
        ax.imshow(rep, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
        for i in range(rep.shape[0]):
            for j in range(rep.shape[1]):
                ax.text(j, i, f"{rep[i, j]:.2f}", ha="center", va="center", fontsize=10,
                        color="white" if rep[i, j] > 0.6 else "black")
        ax.set_xticks(range(3), [translate(c, LANG) for c in ("precision", "recall", "f1-score")])
        ax.set_yticks(range(4), ["0", "1", translate("accuracy", LANG), translate("macro avg", LANG)])
        ax.set_xlabel(translate("Metrics", LANG)); ax.set_ylabel(translate("Classes", LANG))
        ax.set_title(translate(title, LANG) + f" ({translate('risk cut', LANG)} {cut:.4f})", fontsize=9)
        ax2 = axes[row][1]
        ConfusionMatrixDisplay(confusion_matrix=np.array([[tn, fp], [fn, tp]]), display_labels=[0, 1]).plot(
            ax=ax2, cmap="Blues", colorbar=False, values_format=",d")
        ax2.set_xlabel(translate("Predicted label", LANG)); ax2.set_ylabel(translate("True label", LANG))
        ax2.set_title(translate("Confusion matrix", LANG), fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / (fname.replace(".png", "_es.png") if LANG == "es" else fname), dpi=200); plt.show()

# ROC curves of every learner on the 20% comparison part
plot_roc([(k, y_te, p) for k, p in pred.items()], "Preliminary run: comparison part (20% of development)", "04_roc_learners.png")

## Notebook summary

Selection and metrics of each learner, including accuracy and F1 at the top 10% of risk.

In [ ]:
# Summary of notebook 04
sel_s = pd.Series({"primary learner (rule)": primary, "best base learner by AUROC": best,
                   "ensemble adopted": bool(adopt), "native-XGBoost diagnostic minus XGBoost (AUROC)":
                   round(res.loc["XGB native missing (diagnostic)", "AUROC"] - res.loc["XGB", "AUROC"], 4)}, name="value")
display(translate(sel_s.to_frame(), LANG))
cut10 = {k: cut_top(p, w_te, 0.10) for k, p in pred.items()}
summary = res.join(pd.DataFrame({k: threshold_metrics(y_te, p, w_te, cut10[k]) for k, p in pred.items()}).T[["accuracy", "F1", "sensitivity", "specificity"]])
translate(summary, LANG)